<a href="https://colab.research.google.com/github/BardRimon/Study/blob/main/InformationExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задачи

5. Создать пайплайн с оценкой качества
6. Применить методы NER из прошлого курса (морфологические анализаторы и открытые библиотеки) и оценить их по показателям SemEval
7. Применить модели, обученные на FactRuEval (deeppavlov и др.), на размеченном корпусе с оценкой качества выделения сущностей, например, локация и организация
8. Сравнить качество существующих моделей и последних LLM

1. Скачать тестовую часть NEREL.
2. Применить LLM для извлечения вложенных именованных сущностей и отношений между ними в режиме 0- и 5-shot промптинга с оценкой качества.


## тестовый прогон

In [8]:
# @title 0. Cкачивание nerel
!git clone https://github.com/nerel-ds/NEREL.git

Cloning into 'NEREL'...
remote: Enumerating objects: 3078, done.
remote: Counting objects: 100% (3078/3078), done.
remote: Compressing objects: 100% (2896/2896), done.
remote: Total 3078 (delta 1159), reused 2072 (delta 179), pack-reused 0 (from 0)
Receiving objects: 100% (3078/3078), 3.23 MiB | 4.38 MiB/s, done.
Resolving deltas: 100% (1159/1159), done.


In [12]:
import os
import json
from collections import defaultdict

# 1. Настройки путей (проверьте, где лежит скачанная папка)
# Если вы сделали git clone, путь скорее всего такой:
NEREL_PATH = '/content/NEREL/NEREL-v1.1/train'  # Или ./nerel/train, проверьте имя папки через !ls
OUTPUT_FILE = 'dataset.json'

# 2. Маппинг (Словарь перевода тегов NEREL в ваши поля)
# В NEREL теги на английском (PERSON, ORGANIZATION), а ваш код ждет "Кто", "Стоимость"
TAG_MAPPING = {
    'PERSON': 'Кто',
    'ORGANIZATION': 'Кто',      # Организации тоже могут быть субъектами
    'MONEY': 'Стоимость',
    'DATE': 'Дата',
    'COUNTRY': 'Регион',
    'CITY': 'Регион',
    'STATE_OR_PROVINCE': 'Регион',
    # 'EVENT': 'Цель'           # Можно попробовать мапить события на Цель, но это не всегда точно
}

def parse_ann_file(ann_path, txt_content):
    """Парсит .ann файл и извлекает сущности по оффсетам."""
    entities = defaultdict(list)

    with open(ann_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if not parts[0].startswith('T'): continue # Нас интересуют только сущности (T-tags)

            # parts[1] выглядит как "PERSON 10 20"
            entity_info = parts[1].split()
            tag = entity_info[0]

            # Если этот тег нам интересен (есть в маппинге)
            if tag in TAG_MAPPING:
                # Извлекаем текст сущности (parts[2])
                entity_text = parts[2]

                # Записываем в наш словарь под нужным русским ключом
                my_key = TAG_MAPPING[tag]
                entities[my_key].append(entity_text)

    return entities

# 3. Основной цикл конвертации
converted_data = []

if not os.path.exists(NEREL_PATH):
    print(f"Ошибка: Папка {NEREL_PATH} не найдена. Проверьте путь (сделайте !ls)")
else:
    files = [f for f in os.listdir(NEREL_PATH) if f.endswith('.txt')]
    print(f"Найдено {len(files)} файлов. Начинаю обработку...")

    for txt_file in files:
        base_name = txt_file[:-4]
        txt_path = os.path.join(NEREL_PATH, txt_file)
        ann_path = os.path.join(NEREL_PATH, base_name + '.ann')

        if not os.path.exists(ann_path): continue

        # Читаем текст
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Извлекаем сущности
        labels = parse_ann_file(ann_path, text)

        # Добавляем пустые списки для полей, которых не нашли (чтобы код не падал)
        for key in ["Кто", "Тип вложения", "Цель", "Регион", "Стоимость", "Дата"]:
            if key not in labels:
                labels[key] = []

        # Собираем элемент
        converted_data.append({
            "text": text,
            "labels": labels
        })

    # 4. Сохраняем результат
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(converted_data, f, ensure_ascii=False, indent=4)

    print(f"Готово! Создан файл {OUTPUT_FILE} с {len(converted_data)} примерами.")
    print("Теперь загрузите его в ячейке 2 вместо старого кода.")

Найдено 746 файлов. Начинаю обработку...
Готово! Создан файл dataset.json с 746 примерами.
Теперь загрузите его в ячейке 2 вместо старого кода.


In [9]:
# @title 1. Установка зависимостей
# Устанавливаем Natasha для правил, Transformers для нейросетей
!pip install -q natasha pymorphy2 transformers accelerate bitsandbytes scikit-learn
print("Установка завершена.")

Установка завершена.


In [13]:
# @title 2. Подготовка данных (Обновленная)
import json
# ... импорты метрик ...

# ЗАГРУЗКА РЕАЛЬНОГО ДАТАСЕТА
try:
    with open('dataset.json', 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print(f"Успешно загружен датасет NEREL: {len(raw_data)} документов.")
except FileNotFoundError:
    print("Файл dataset.json не найден! Сначала выполните скрипт конвертации.")
    # Тут можно оставить фоллбэк на игрушечные данные, если хотите

# Дальше функция calculate_metrics без изменений...

# --- 2. Метрики (Strict & Partial) ---
def calculate_metrics(pred_data, true_data):
    """Считает точность (Precision), полноту (Recall) и F1."""
    tp_strict, tp_partial, fp, fn = 0, 0, 0, 0

    for i in range(len(pred_data)):
        pred = pred_data[i]
        true = true_data[i]

        # Нормализация
        p_set = set([str(x).lower().strip() for x in pred if x])
        t_set = set([str(x).lower().strip() for x in true if x])

        if not t_set and not p_set: continue # Оба пустые - ок

        # Strict match (полное совпадение)
        if p_set == t_set:
            tp_strict += 1

        # Partial match (пересечение слов)
        # Если есть хотя бы одно общее слово (грубая оценка)
        elif any(word in ' '.join(t_set) for p_str in p_set for word in p_str.split()):
            tp_partial += 1
        elif p_set and not t_set:
            fp += 1
        elif t_set and not p_set:
            fn += 1
        else:
            # Если не совпало совсем
            fp += 1
            fn += 1

    # Расчет для Partial (считаем частичное как 0.5 успеха)
    total_relevant = len(pred_data) # Упрощенно для примера
    accuracy = (tp_strict + 0.5 * tp_partial) / (tp_strict + tp_partial + fp + fn + 1e-9)
    return round(accuracy, 2)

print("Данные загружены. Количество примеров:", len(raw_data))

Успешно загружен датасет NEREL: 746 документов.
Данные загружены. Количество примеров: 746


In [15]:
# @title 3. Метод Natasha (Rule-based)
import re
from natasha import (
    Segmenter, MorphVocab,
    NewsEmbedding, NewsNERTagger,
    Doc
)
import pymorphy2

class NatashaExtractor:
    def __init__(self):
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(self.emb)
        self.morph = pymorphy2.MorphAnalyzer()

    def extract(self, text):
        doc = Doc(text)
        doc.segment(self.segmenter)
        doc.tag_ner(self.ner_tagger)

        # 1. Извлечение Сумм (Regex)
        money_regex = r'(\d+(?:[\.,]\d+)?\s*(?:млрд|млн|тыс)?\s*(?:руб|доллар|евро)[а-я]*)'
        cost = re.findall(money_regex, text, re.IGNORECASE)

        # 2. Извлечение Дат (Regex + Natasha)
        date_regex = r'(\d{4}\s*год[а-у]?)'
        date = re.findall(date_regex, text, re.IGNORECASE)

        # 3. Извлечение Организаций и Локаций (Natasha NER)
        who = []
        region = []
        for span in doc.spans:
            if span.type == 'ORG':
                who.append(span.text)
            elif span.type == 'LOC':
                region.append(span.text)

        # 4. Тип вложения (Лемматизация глаголов)
        verbs = {'инвестировать', 'выделить', 'вложить', 'направить'}
        actions = []
        for token in doc.tokens:
            p = self.morph.parse(token.text)[0]
            if p.normal_form in verbs:
                actions.append(token.text)

        return {
            "Кто": who,
            "Тип вложения": actions,
            "Цель": [], # Сложно для правил
            "Регион": region,
            "Стоимость": cost,
            "Дата": date
        }

# Тест
extractor_natasha = NatashaExtractor()
res = extractor_natasha.extract(raw_data[0]['text'])
print("Natasha результат:", res)

Natasha результат: {'Кто': ['ФИДЕ', 'ФИДЕ', 'ФИДЕ'], 'Тип вложения': [], 'Цель': [], 'Регион': ['Элисте', 'Дортмунде'], 'Стоимость': [], 'Дата': ['1975 года', '1975 года', '2005 год', '2005 году']}


In [18]:
# @title 4. Метод BERT (Исправленный)
from transformers import pipeline

class BertNerExtractor:
    def __init__(self):
        # ИСПОЛЬЗУЕМ МОДЕЛЬ, УЖЕ ОБУЧЕННУЮ НА NER
        # Babelscape - мощная мультиязычная модель, понимающая PER, LOC, ORG
        self.ner_pipeline = pipeline(
            "ner",
            model="Babelscape/wikineural-multilingual-ner",
            aggregation_strategy="simple"
        )

    def extract(self, text):
        # Ограничим длину текста, чтобы модель не упала (BERT лимит 512 токенов)
        # pipeline обычно сам обрезает, но лучше перестраховаться
        safe_text = text[:2000]

        try:
            ner_results = self.ner_pipeline(safe_text)
        except Exception as e:
            print(f"Ошибка BERT: {e}")
            return {"error": str(e)}

        who = []
        region = []

        # Маппинг тегов (PER/ORG -> Кто, LOC -> Регион)
        for entity in ner_results:
            # entity_group выдает обобщенные теги (PER, ORG, LOC, MISC)
            tag = entity.get('entity_group', 'MISC')
            word = entity.get('word', '')

            if tag in ['ORG', 'PER']:
                who.append(word)
            elif tag == 'LOC':
                region.append(word)

        return {
            "Кто": list(set(who)),       # убираем дубликаты
            "Тип вложения": [],
            "Цель": [],
            "Регион": list(set(region)),
            "Стоимость": [],
            "Дата": []
        }

# Тест
print("Загружаю BERT (Babelscape)...")
extractor_bert = BertNerExtractor()
# Берем текст из загруженного датасета, если он есть, иначе тестовый
test_txt = raw_data[0]['text'] if 'raw_data' in globals() and raw_data else "Apple инвестировала в Москве."
res = extractor_bert.extract(test_txt)
print("BERT результат:", res)

Загружаю BERT (Babelscape)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


BERT результат: {'Кто': ['Владимир Крамник', 'Каспаров', 'Гарри Каспарова', 'Топалов', 'Найджел Шорт', 'Гарри Каспаров', 'Веселину Топалову', 'Крамник', 'ФИДЕ', 'Гари Каспаров', 'Яном Тимманом', 'Краник', 'Анатолием Карповым', 'Веселин Топалов'], 'Тип вложения': [], 'Цель': [], 'Регион': ['Элисте'], 'Стоимость': [], 'Дата': []}


In [19]:
# @title 5. Метод LLM (Qwen - без токена)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json

# Проверка GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

class LLMExtractor:
    def __init__(self):
        # Qwen2.5-1.5B-Instruct - открытая модель, не нужен токен HF
        model_name = "Qwen/Qwen2.5-1.5B-Instruct"

        print(f"Загрузка модели {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Загружаем модель
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16, # Используем float16 для GPU
            device_map="auto"
        )

    def extract(self, text):
        # Промпт специально для Qwen (формат ChatML)
        messages = [
            {"role": "system", "content": "Ты полезный ассистент, эксперт по извлечению информации."},
            {"role": "user", "content": f"""
            Проанализируй текст и извлеки информацию в формате JSON с полями:
            - "Кто" (организации или персоны)
            - "Тип вложения" (глагол, например: инвестировал, выделил)
            - "Цель" (на что пошли деньги)
            - "Регион" (географическое место)
            - "Стоимость" (сумма денег)
            - "Дата" (год или время)

            Если поля нет в тексте, оставь пустой список [].
            Не пиши ничего лишнего, только JSON.

            Текст: {text[:2000]}
            """}
        ]

        # Подготовка ввода
        text_input = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([text_input], return_tensors="pt").to(device)

        # Генерация
        generated_ids = self.model.generate(
            model_inputs.input_ids,
            max_new_tokens=256,
            do_sample=False,   # Жадная генерация для воспроизводимости
            temperature=0.0    # Минимальная температура
        )

        # Декодирование (убираем сам промпт из ответа)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Попытка найти и распарсить JSON
        try:
            # Ищем границы JSON {}
            start = response.find('{')
            end = response.rfind('}') + 1
            if start != -1 and end != 0:
                json_str = response[start:end]
                return json.loads(json_str)
            else:
                return {"error": "No JSON found", "raw": response}
        except Exception as e:
            return {"error": "JSON parse failed", "raw": response}

# Запуск
try:
    extractor_llm = LLMExtractor()
    # Тест на первом примере
    test_txt = raw_data[0]['text'] if 'raw_data' in globals() and raw_data else "Компания Apple инвестировала 1 млрд."
    res = extractor_llm.extract(test_txt)
    print("\nLLM Результат:", res)
except Exception as e:
    print("Ошибка при запуске LLM:", e)

Используемое устройство: cuda
Загрузка модели Qwen/Qwen2.5-1.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



LLM Результат: {'error': 'JSON parse failed', 'raw': '```json\n[\n    {\n        "Кто": ["Гарри Каспаров", "ФИДЕ", "Веселин Топалов"],\n        "Тип вложения": [],\n        "Цель": ["предоставить информацию о матче на первенстве мира по шахматам"],\n        "Регион": ["Элиста"],\n        "Стоимость": [],\n        "Дата": ["1993"]\n    },\n    {\n        "Кто": ["Владимир Крамник", "Чемпион мира ФИДЕ"],\n        "Тип вложения": [],\n        "Цель": ["предоставить информацию о матче на первенстве мира по шахматам"],\n        "Регион": ["Элиста"],\n        "Стоимость": [],\n        "Дата": ["1993"]\n    }\n]\n```'}


In [20]:
# @title 6. Итоговое сравнение (Оптимизированное GPU)
from tqdm.auto import tqdm # Полоса прогресса
import pandas as pd

# 1. Настройки
TEST_SIZE = 50  # Количество примеров для теста (поставьте None, если хотите весь датасет)
BATCH_SIZE = 16 # Размер пачки для GPU (если падает с ошибкой памяти, уменьшите до 8 или 4)

# Берем подмножество данных или все данные
data_slice = raw_data[:TEST_SIZE] if TEST_SIZE else raw_data
texts = [d['text'] for d in data_slice]
true_labels_list = [d['labels'] for d in data_slice]

print(f"Обработка {len(texts)} документов...")

# --- 2. Запуск NATASHA (CPU, последовательно) ---
print("1/3 Запуск Natasha...")
natasha_preds = []
for text in tqdm(texts, desc="Natasha"):
    natasha_preds.append(extractor_natasha.extract(text))

# --- 3. Запуск BERT (GPU, пакетно - Исправление вашей ошибки) ---
print("2/3 Запуск BERT (Batch mode)...")
bert_preds = []

# Используем pipeline прямо на списке текстов для эффективности
# (Мы обращаемся к pipeline внутри класса, чтобы использовать batch_size)
try:
    # Получаем сырые результаты батчами
    raw_bert_results = extractor_bert.ner_pipeline(texts, batch_size=BATCH_SIZE)

    # Обрабатываем результаты по мере поступления
    for ner_results in tqdm(raw_bert_results, total=len(texts), desc="BERT"):
        who = []
        region = []

        # Логика парсинга (та же, что была в классе)
        for entity in ner_results:
            tag = entity.get('entity_group', 'MISC')
            word = entity.get('word', '')
            if tag in ['ORG', 'PER']:
                who.append(word)
            elif tag == 'LOC':
                region.append(word)

        bert_preds.append({
            "Кто": list(set(who)),
            "Тип вложения": [],
            "Цель": [],
            "Регион": list(set(region)),
            "Стоимость": [],
            "Дата": []
        })
except Exception as e:
    print(f"Ошибка в BERT pipeline: {e}")
    # Фолбэк (пустые результаты)
    bert_preds = [{}] * len(texts)

# --- 4. Запуск LLM (GPU, последовательно для безопасности) ---
# LLM очень тяжелая, batching на бесплатном Colab может вызвать OutOfMemory.
# Лучше оставить последовательно, но с прогресс-баром.
print("3/3 Запуск LLM...")
llm_preds = []

# Если LLM загружена (проверка на случай ошибок в ячейке 5)
if 'extractor_llm' in globals():
    for text in tqdm(texts, desc="LLM"):
        try:
            # Обрезаем текст, чтобы не переполнить контекст
            res = extractor_llm.extract(text[:1500])
            # Если вернулась ошибка словарем, заменяем на пустой dict для метрик
            if "error" in res:
                llm_preds.append({})
            else:
                llm_preds.append(res)
        except Exception as e:
            llm_preds.append({})
else:
    print("LLM не инициализирована, пропускаем.")
    llm_preds = [{}] * len(texts)

# --- 5. Сводная таблица и Метрики ---
results_table = []

print("Расчет метрик...")
for i in range(len(texts)):
    t_labels = true_labels_list[i]

    # Метрики считаем по полю "Кто" (или "Регион") как основному индикатору
    # Для NEREL это поля PER/ORG
    score_natasha = calculate_metrics([natasha_preds[i].get("Кто", [])], [t_labels.get("Кто", [])])
    score_bert = calculate_metrics([bert_preds[i].get("Кто", [])], [t_labels.get("Кто", [])])
    score_llm = calculate_metrics([llm_preds[i].get("Кто", [])], [t_labels.get("Кто", [])])

    results_table.append({
        "Text Snippet": texts[i][:50] + "...",
        "Natasha F1": score_natasha,
        "BERT F1": score_bert,
        "LLM F1": score_llm,
        "LLM Output": str(llm_preds[i].get("Кто", []))[:50] # Для отладки
    })

df_res = pd.DataFrame(results_table)

# Вывод средних показателей
print("\n=== ИТОГОВЫЕ РЕЗУЛЬТАТЫ (Средний Score) ===")
print(f"Natasha: {df_res['Natasha F1'].mean():.2f}")
print(f"BERT:    {df_res['BERT F1'].mean():.2f}")
print(f"LLM:     {df_res['LLM F1'].mean():.2f}")

display(df_res.head(10))

Обработка 50 документов...
1/3 Запуск Natasha...


Natasha:   0%|          | 0/50 [00:00<?, ?it/s]

2/3 Запуск BERT (Batch mode)...


BERT:   0%|          | 0/50 [00:00<?, ?it/s]

3/3 Запуск LLM...


LLM:   0%|          | 0/50 [00:00<?, ?it/s]

Расчет метрик...

=== ИТОГОВЫЕ РЕЗУЛЬТАТЫ (Средний Score) ===
Natasha: 0.43
BERT:    0.54
LLM:     0.33


,Text Snippet,Natasha F1,BERT F1,LLM F1,LLM Output
0,Матч Топалов — Крамник (23 сентября)\nСегодня ...,0.5,1.0,0.0,[]
1,Мать Тереза стала Святой Терезой Калькуттской\...,0.5,0.5,0.0,[]
2,Бостон взорвали Тамерлан и Джохар Царнаевы из ...,0.5,0.5,0.5,"['Джохар Царнаев', 'Тамерлан Царнаев']"
3,Муртаза Рахимов чистит кадры\n\t\nПрезидент Ба...,0.5,0.5,0.0,[]
4,Максим Орешкин назначен министром экономическо...,0.5,0.5,0.5,['Максим Орешкин']
5,У Бооса родился сын\n\nГеоргий Боос\nУ бывшего...,0.5,0.5,0.5,['Георгий Боос']
6,Святейший Патриарх Кирилл прибыл в Санкт-Петер...,0.5,0.5,0.5,Святейший Патриарх Кирилл
7,Боксёр Тайсон Фьюри признался в употреблении к...,0.5,1.0,0.5,Боксёр Тайсон Фьюри
8,Москве потребовался год на поиски кандидата в ...,0.5,0.5,0.0,[]
9,Предполагаемый план создания радиационной пушк...,0.5,0.5,0.5,"['Глендон Скотт Кроуфорд', 'Эрик Фейт']"


Results of nested NER for NEREL
Method	P	R	F1
Biaffine, fT	81.64	77.69	79.62
Biaffine, RuBERT, ft	80.71	77.84	79.25
Pyramid, fT	75.87	72.40	74.09
Pyramid, RuBERT, ft	79.54	79.91	79.73
SpERT, RuBERT	82.90	82.14	82.52
MRC	85.04	84.95	84.99


## попытка 2

### Пункт 5 pipline

In [26]:
import abc
from dataclasses import dataclass
from typing import List, Set, Dict, Tuple, Any, Optional
from collections import defaultdict
# Сторонние библиотеки
import torch
from transformers import pipeline
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsNERTagger,
    NamesExtractor,
    DatesExtractor,
    MoneyExtractor,
    AddrExtractor,
    Doc
)

# ==========================================
# 1. Единый формат данных (Data Contract)
# ==========================================

@dataclass
class Entity:
    """
    Универсальное представление сущности.
    """
    text: str          # Текст сущности
    label: str         # Тип (PER, ORG, LOC, DATE, MONEY)
    start: int         # Начальный индекс в тексте
    end: int           # Конечный индекс
    confidence: Optional[float] = None  # Уверенность модели (если есть)

# ==========================================
# 2. Абстрактный базовый класс
# ==========================================

class BaseNERPipeline(abc.ABC):
    """
    Интерфейс для всех NER-экстракторов.
    """

    @abc.abstractmethod
    def extract(self, text: str) -> List[Entity]:
        """
        Основной метод извлечения сущностей.
        """
        pass

# ==========================================
# 3. Реализация Сценария А: Natasha (Rules + Embeddings)
# ==========================================

class NatashaPipeline(BaseNERPipeline):
    def __init__(self):
        # Инициализация компонентов Natasha
        # Это "ленивая" загрузка, выполняется один раз при старте
        self.segmenter = Segmenter()
        self.morph_vocab = MorphVocab()

        # Эмбеддинги для NER (names, orgs, locs)
        emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(emb)

        # Экстракторы на правилах (очень точные для дат и денег)
        self.dates_extractor = DatesExtractor(self.morph_vocab)
        self.money_extractor = MoneyExtractor(self.morph_vocab)
        self.names_extractor = NamesExtractor(self.morph_vocab) # Для нормализации имен

    def extract(self, text: str) -> List[Entity]:
        doc = Doc(text)

        # 1. Сегментация (разбиение на токены и предложения)
        doc.segment(self.segmenter)

        # 2. NER через нейросеть (Slovnet внутри Natasha)
        doc.tag_ner(self.ner_tagger)

        # 3. Нормализация (приведение к начальной форме)
        for span in doc.spans:
            span.normalize(self.morph_vocab)

        entities = []

        # Конвертация спанов Natasha в наш формат Entity
        for span in doc.spans:
            entities.append(Entity(
                text=span.normal, # Или span.text для исходного
                label=span.type,  # PER, LOC, ORG
                start=span.start,
                end=span.stop,
                confidence=1.0    # Natasha не отдает confidence явно в простом API
            ))

        # 4. Дополнительное извлечение фактов (Правила)
        # Пример для Денег (Money)
        money_matches = self.money_extractor(text)
        for match in money_matches:
            entities.append(Entity(
                text=text[match.start:match.stop],
                label='MONEY',
                start=match.start,
                end=match.stop,
                confidence=1.0
            ))

        # Сортируем по появлению в тексте
        return sorted(entities, key=lambda x: x.start)

# ==========================================
# 4. Реализация Сценария B: Hugging Face (Transformers)
# ==========================================

class TransformersPipeline(BaseNERPipeline):
    def __init__(self, model_name: str = "Babelscape/wikineural-multilingual-ner", device: int = -1):
        """
        Args:
            model_name: Имя модели. По умолчанию используем проверенную Babelscape.
            device: -1 для CPU, 0 для GPU.
        """
        print(f"Loading transformer model: {model_name}...")
        try:
            self.pipeline = pipeline(
                "ner",
                model=model_name,
                tokenizer=model_name,
                aggregation_strategy="simple", # Склеивает токены (Влад + ##имир -> Владимир)
                device=device
            )
        except OSError as e:
            print(f"CRITICAL ERROR: Не удалось загрузить модель '{model_name}'.")
            print("Проверь подключение к интернету или правильность имени модели на https://huggingface.co/models")
            raise e

    def extract(self, text: str) -> List[Entity]:
        # Инференс
        try:
            raw_results = self.pipeline(text)
        except Exception as e:
            print(f"Error during inference: {e}")
            return []

        entities = []
        for item in raw_results:
            # У этой модели теги приходят в формате 'PER', 'LOC', 'ORG' (без B-/I-)
            # thanks to aggregation_strategy="simple"

            # Фильтруем мусор, если уверенность низкая (опционально)
            if item['score'] < 0.30:
                continue

            entities.append(Entity(
                text=item['word'],
                label=item['entity_group'], # Babelscape возвращает entity_group
                start=item['start'],
                end=item['end'],
                confidence=float(item['score'])
            ))

        # Сортировка по порядку появления в тексте
        return sorted(entities, key=lambda x: x.start)



### Тест pipline

In [25]:
# ==========================================
# 5. Демонстрация (Client Code)
# ==========================================

if __name__ == "__main__":
    text = "Иван Петров купил акции Газпрома в Москве за 10000 рублей 5 мая 2023 года."

    print(f"Исходный текст: {text}\n")

    # Сценарий A: Natasha
    print("--- Natasha Extraction ---")
    natasha_pipe = NatashaPipeline()
    for ent in natasha_pipe.extract(text):
        print(f"[{ent.label}] {ent.text} ({ent.start}-{ent.end})")

    print("\n--- Transformers Extraction (ruBert-tiny2) ---")
    # Используем CPU для демо
    hf_pipe = TransformersPipeline(device=-1)
    for ent in hf_pipe.extract(text):
        print(f"[{ent.label}] {ent.text} (conf: {ent.confidence:.2f})")

Исходный текст: Иван Петров купил акции Газпрома в Москве за 10000 рублей 5 мая 2023 года.

--- Natasha Extraction ---
[PER] Иван Петров (0-11)
[ORG] Газпрома (24-32)
[LOC] Москве (35-41)
[MONEY] 10000 рублей 5 (45-59)

--- Transformers Extraction (ruBert-tiny2) ---
Loading transformer model: Babelscape/wikineural-multilingual-ner...


Device set to use cpu


[PER] Иван Петров (conf: 1.00)
[ORG] Газпрома (conf: 0.96)
[LOC] Москве (conf: 1.00)


In [27]:
from dataclasses import dataclass
from typing import List, Set, Dict, Tuple, Any
from collections import defaultdict

# ==========================================
# 1. Обновленный Data Contract (Добавили eq/hash)
# ==========================================

@dataclass(frozen=True) # frozen=True делает объект неизменяемым и хешируемым
class Entity:
    text: str
    label: str
    start: int
    end: int
    confidence: float = 1.0

    # Метод для проверки пересечения спанов (для Partial matching)
    def intersects(self, other: 'Entity') -> bool:
        # Проверка пересечения отрезков [start, end]
        return (self.start < other.end) and (self.end > other.start)

# ==========================================
# 2. Класс для расчета метрик (SemEval style)
# ==========================================

@dataclass
class EvaluationMetrics:
    precision: float
    recall: float
    f1: float
    support: int  # Количество сущностей в эталоне

class NEREvaluator:
    """
    Класс для расчета метрик качества NER:
    - Strict Metrics (SemEval): точное совпадение границ и типа.
    - Partial Metrics: частичное пересечение границ + совпадение типа.
    """

    def evaluate(self, true_entities: List[Entity], pred_entities: List[Entity]) -> Dict[str, Any]:
        """
        Основной метод оценки.
        Возвращает словарь с метриками (общие + по классам).
        """
        # 1. Рассчитываем Strict Metrics (Строгие)
        strict_res = self._compute_metrics(true_entities, pred_entities, mode='strict')

        # 2. Рассчитываем Partial Metrics (Мягкие)
        partial_res = self._compute_metrics(true_entities, pred_entities, mode='partial')

        return {
            "strict": strict_res,
            "partial": partial_res
        }

    def _compute_metrics(self, y_true: List[Entity], y_pred: List[Entity], mode: str = 'strict') -> Dict[str, Any]:
        """
        Внутренняя логика подсчета TP, FP, FN.
        """
        # Счетчики: Global
        tp, fp, fn = 0, 0, 0

        # Счетчики: Per Class
        class_stats = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

        # Используем списки, чтобы помечать "использованные" сущности (чтобы не мапить одну сущность дважды)
        pred_matched = [False] * len(y_pred)
        true_matched = [False] * len(y_true)

        # 1. Ищем True Positives (TP)
        for i, pred in enumerate(y_pred):
            for j, true in enumerate(y_true):
                if true_matched[j]: continue # Эта сущность уже найдена

                is_match = False
                if mode == 'strict':
                    # Полное совпадение границ и лейбла
                    if pred.start == true.start and pred.end == true.end and pred.label == true.label:
                        is_match = True
                elif mode == 'partial':
                    # Пересечение границ и совпадение лейбла
                    if pred.intersects(true) and pred.label == true.label:
                        is_match = True

                if is_match:
                    tp += 1
                    class_stats[true.label]['tp'] += 1
                    pred_matched[i] = True
                    true_matched[j] = True
                    break # Переходим к следующему предикту

        # 2. Ищем False Positives (FP) - предсказали то, чего нет
        for i, matched in enumerate(pred_matched):
            if not matched:
                fp += 1
                class_stats[y_pred[i].label]['fp'] += 1

        # 3. Ищем False Negatives (FN) - пропустили то, что было
        for j, matched in enumerate(true_matched):
            if not matched:
                fn += 1
                class_stats[y_true[j].label]['fn'] += 1

        # Собираем итоговые метрики
        overall = self._calc_prf(tp, fp, fn)

        per_class = {}
        for label, stats in class_stats.items():
            per_class[label] = self._calc_prf(stats['tp'], stats['fp'], stats['fn'])

        return {
            "overall": overall,
            "per_class": per_class
        }

    def _calc_prf(self, tp: int, fp: int, fn: int) -> EvaluationMetrics:
        """Вспомогательная функция для формул"""
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        return EvaluationMetrics(
            precision=round(precision, 4),
            recall=round(recall, 4),
            f1=round(f1, 4),
            support=tp + fn
        )

# ==========================================
# 3. Пример использования (Integration Test)
# ==========================================

if __name__ == "__main__":
    # 1. Ground Truth (Эталон)
    # Текст: "Иван купил билет в Москву"
    true_entities = [
        Entity("Иван", "PER", 0, 4),
        Entity("Москву", "LOC", 19, 25)
    ]

    # 2. Prediction (Гипотеза модели)
    # Допустим, модель немного ошиблась в границах и нашла лишнее
    pred_entities = [
        Entity("Иван", "PER", 0, 4),           # Идеальное совпадение (Strict TP)
        Entity("в Москву", "LOC", 17, 25),    # Ошибка границ (Partial TP, Strict FP/FN)
        Entity("билет", "OBJ", 11, 16)        # Лишняя сущность (FP)
    ]

    evaluator = NEREvaluator()
    metrics = evaluator.evaluate(true_entities, pred_entities)

    print("=== Результаты оценки ===")

    print("\n[Strict Metrics] (Жесткие требования SemEval):")
    m = metrics['strict']['overall']
    print(f"Precision: {m.precision}, Recall: {m.recall}, F1: {m.f1}")

    print("\n[Partial Metrics] (Допускается пересечение границ):")
    m = metrics['partial']['overall']
    print(f"Precision: {m.precision}, Recall: {m.recall}, F1: {m.f1}")

    print("\n[Per Class Strict] (По классам):")
    for label, res in metrics['strict']['per_class'].items():
        print(f"Class {label}: F1 = {res.f1}")

=== Результаты оценки ===

[Strict Metrics] (Жесткие требования SemEval):
Precision: 0.3333, Recall: 0.5, F1: 0.4

[Partial Metrics] (Допускается пересечение границ):
Precision: 0.6667, Recall: 1.0, F1: 0.8

[Per Class Strict] (По классам):
Class PER: F1 = 1.0
Class LOC: F1 = 0.0
Class OBJ: F1 = 0.0


In [28]:
import os
from typing import List, Dict
from tqdm.auto import tqdm # Прогресс-бар

# Импортируем Natasha
from natasha import (
    Segmenter, MorphVocab, NewsEmbedding,
    NewsNERTagger, Doc
)

# ==========================================
# 1. Конфигурация Маппинга (NEREL -> Standard)
# ==========================================

# Превращаем детальные теги NEREL в общие, которые понимает Natasha
NEREL_TO_STD_MAP = {
    'PERSON': 'PER',
    'ORGANIZATION': 'ORG',
    'COUNTRY': 'LOC',
    'CITY': 'LOC',
    'STATE_OR_PROVINCE': 'LOC',
    'LOCATION': 'LOC',
    'FACILITY': 'LOC' # Иногда здания размечают как FACility, для Natasha это LOC/ORG
}

# ==========================================
# 2. Загрузчик данных (NEREL Parser)
# ==========================================

class NerelLoader:
    def __init__(self, dataset_path: str):
        self.dataset_path = dataset_path

    def load(self) -> List[Dict]:
        """
        Читает пары .txt и .ann, возвращает список словарей:
        [{'text': str, 'entities': List[Entity]}, ...]
        """
        documents = []

        # Получаем список файлов (убираем расширение)
        filenames = set()
        for f in os.listdir(self.dataset_path):
            if f.endswith('.txt'):
                filenames.add(f[:-4])

        print(f"Найдено документов: {len(filenames)}")

        for name in tqdm(filenames, desc="Loading NEREL"):
            txt_path = os.path.join(self.dataset_path, name + '.txt')
            ann_path = os.path.join(self.dataset_path, name + '.ann')

            # 1. Читаем текст
            with open(txt_path, 'r', encoding='utf-8') as f:
                text = f.read()

            # 2. Читаем аннотации
            entities = []
            if os.path.exists(ann_path):
                with open(ann_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if not line.startswith('T'): continue # Нас интересуют только Text-bound entities

                        # Формат: T1  PERSON 10 20  Иван
                        parts = line.split('\t')
                        if len(parts) < 3: continue

                        tag_info = parts[1].split() # 'PERSON 10 20' или 'PERSON 10 15;16 20'
                        nerel_tag = tag_info[0]

                        # Парсим оффсеты (обрабатываем разрывные спаны, беря min и max)
                        # Пример: "10 15;16 20" -> start=10, end=20
                        spans = []
                        for s in tag_info[1:]:
                            if ';' in s:
                                sub_s = s.split(';')
                                spans.extend(sub_s)
                            else:
                                spans.append(s)

                        try:
                            start = int(spans[0])
                            end = int(spans[-1])
                        except ValueError:
                            continue # Пропускаем битую разметку

                        # Маппинг и фильтрация
                        # Если тег есть в нашем маппинге - берем, иначе пропускаем (т.к. Natasha его не знает)
                        if nerel_tag in NEREL_TO_STD_MAP:
                            mapped_tag = NEREL_TO_STD_MAP[nerel_tag]
                            entities.append(Entity(
                                text=parts[2],
                                label=mapped_tag,
                                start=start,
                                end=end
                            ))

            documents.append({
                "text": text,
                "entities": sorted(entities, key=lambda x: x.start)
            })

        return documents

# ==========================================
# 3. Реализация Natasha Pipeline (из прошлого шага)
# ==========================================

class NatashaBaseline:
    def __init__(self):
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(self.emb)
        self.morph_vocab = MorphVocab() # Для нормализации (опционально)

    def extract(self, text: str) -> List[Entity]:
        doc = Doc(text)
        doc.segment(self.segmenter)
        doc.tag_ner(self.ner_tagger)

        entities = []
        for span in doc.spans:
            # Natasha выдает PER, ORG, LOC - они уже соответствуют нашему стандарту
            entities.append(Entity(
                text=span.text,
                label=span.type,
                start=span.start,
                end=span.stop
            ))
        return entities

# ==========================================
# 4. Основной скрипт выполнения
# ==========================================

def run_baseline_evaluation():
    # Путь к данным (из твоего запроса)
    DATA_PATH = "/content/NEREL/NEREL-v1.1/test"

    # 1. Загрузка
    loader = NerelLoader(DATA_PATH)
    data = loader.load()

    if not data:
        print("Ошибка: Данные не загружены. Проверь путь.")
        return

    # 2. Инициализация модели
    print("Инициализация Natasha...")
    model = NatashaBaseline()

    # 3. Инициализация оценщика (класс из прошлого ответа)
    evaluator = NEREvaluator()

    # 4. Прогон по всему датасету
    all_true = []
    all_pred = []

    print("Запуск инференса...")
    for doc in tqdm(data, desc="Processing"):
        text = doc['text']
        true_ents = doc['entities']

        # Предсказание
        pred_ents = model.extract(text)

        all_true.extend(true_ents)
        all_pred.extend(pred_ents)

    # 5. Подсчет метрик
    print("\n" + "="*30)
    print("РЕЗУЛЬТАТЫ NATASHA (BASELINE)")
    print("="*30)

    results = evaluator.evaluate(all_true, all_pred)

    # Вывод
    def print_metrics(metrics_dict, title):
        print(f"\n--- {title} ---")
        ov = metrics_dict['overall']
        print(f"[OVERALL] Precision: {ov.precision:.2f} | Recall: {ov.recall:.2f} | F1: {ov.f1:.2f}")
        print("По классам:")
        for label, m in metrics_dict['per_class'].items():
            print(f"  {label:<5}: P={m.precision:.2f} R={m.recall:.2f} F1={m.f1:.2f} (Supp: {m.support})")

    print_metrics(results['strict'], "Strict Match (Строгое совпадение)")
    print_metrics(results['partial'], "Partial Match (Частичное совпадение)")

# Запуск
if __name__ == "__main__":
    # Убедитесь, что Entity и NEREvaluator определены в ноутбуке!
    try:
        run_baseline_evaluation()
    except FileNotFoundError:
        print("Папка с данными не найдена. Проверьте путь.")

Найдено документов: 93


Loading NEREL:   0%|          | 0/93 [00:00<?, ?it/s]

Инициализация Natasha...
Запуск инференса...


Processing:   0%|          | 0/93 [00:00<?, ?it/s]


РЕЗУЛЬТАТЫ NATASHA (BASELINE)

--- Strict Match (Строгое совпадение) ---
[OVERALL] Precision: 0.87 | Recall: 0.71 | F1: 0.78
По классам:
  LOC  : P=0.93 R=0.69 F1=0.80 (Supp: 932)
  ORG  : P=0.72 R=0.57 F1=0.64 (Supp: 675)
  PER  : P=0.90 R=0.84 F1=0.87 (Supp: 961)

--- Partial Match (Частичное совпадение) ---
[OVERALL] Precision: 0.99 | Recall: 0.81 | F1: 0.89
По классам:
  LOC  : P=0.99 R=0.74 F1=0.85 (Supp: 932)
  ORG  : P=0.97 R=0.76 F1=0.85 (Supp: 675)
  PER  : P=0.99 R=0.92 F1=0.95 (Supp: 961)


In [34]:
# @title 7. Исправленная реализация DeepPavlov (RuBERT Collection3)
from transformers import pipeline
import torch
import numpy as np

class DeepPavlovNER(BaseNERPipeline):
    def __init__(self, device: int = 0):
        # Используем модель, обученную на Collection3 (стандарт для русского NER)
        # Это наиболее близкий аналог моделей DeepPavlov
        self.model_name = "viktoroo/sberbank-rubert-base-collection3"
        print(f"Загрузка модели {self.model_name}...")

        try:
            self.pipeline = pipeline(
                "ner",
                model=self.model_name,
                tokenizer=self.model_name,
                aggregation_strategy="simple", # Склеивает токены
                device=device
            )
        except Exception as e:
            print(f"Ошибка загрузки: {e}")
            self.pipeline = None

        # МАППИНГ ТЕГОВ (Самая важная часть исправления)
        # Приводим всё многообразие тегов к трем стандартным
        self.tag_mapping = {
            'PER': 'PER', 'PERSON': 'PER', 'B-PER': 'PER', 'I-PER': 'PER',
            'ORG': 'ORG', 'ORGANIZATION': 'ORG', 'B-ORG': 'ORG', 'I-ORG': 'ORG',
            'LOC': 'LOC', 'LOCATION': 'LOC', 'B-LOC': 'LOC', 'I-LOC': 'LOC',
            'GPE': 'LOC', 'GP': 'LOC' # Гео-политические сущности туда же
        }

    def extract(self, text: str) -> list:
        if not self.pipeline: return []

        # SLIDING WINDOW (Скользящее окно)
        # BERT падает или обрезает тексты длиннее 512 токенов.
        # Мы бьем текст на куски, прогоняем и склеиваем результаты.

        chunk_size = 1000  # Символов (примерно 200-300 токенов)
        overlap = 100      # Перекрытие, чтобы не разрезать сущность посередине

        entities = []
        text_len = len(text)

        # Если текст короткий, обрабатываем целиком
        if text_len <= chunk_size:
            chunks = [(0, text)]
        else:
            chunks = []
            start = 0
            while start < text_len:
                end = min(start + chunk_size, text_len)
                chunk_text = text[start:end]
                chunks.append((start, chunk_text))
                if end == text_len: break
                start += chunk_size - overlap # Сдвигаем окно

        # Прогон по кускам
        for offset, chunk_text in chunks:
            try:
                # Инференс
                results = self.pipeline(chunk_text)

                for item in results:
                    # Получаем тег
                    raw_label = item['entity_group']

                    # Маппинг (Исправление ошибки 0.00)
                    label = self.tag_mapping.get(raw_label)

                    # Если тег не в нашем списке (например, MISC), пропускаем
                    if label:
                        entities.append(Entity(
                            text=item['word'],
                            label=label,
                            start=offset + item['start'], # Adjust start position for chunk
                            end=offset + item['end'],     # Adjust end position for chunk
                            confidence=item['score']
                        ))
            except Exception as e:
                print(f"Error during chunk inference: {e}")
                continue # Continue to the next chunk even if one fails

        # Sort entities by their start position after combining all chunks
        return sorted(entities, key=lambda x: x.start)


In [35]:
def run_improved_comparison():
    # 1. Настройки
    DATA_PATH = "/content/NEREL/NEREL-v1.1/test"
    LIMIT_DOCS = 20  # Меньше документов для быстрого теста

    # 2. Загрузка
    print(">>> Загрузка данных...")
    loader = NerelLoader(DATA_PATH)
    data = loader.load()
    if not data: return
    test_data = data[:LIMIT_DOCS]

    # 3. Инициализация
    print("\n>>> Инициализация модели DeepPavlov (Collection3)...")
    device = 0 if torch.cuda.is_available() else -1
    dp_model = DeepPavlovNER(device=device)

    # ТЕСТОВЫЙ ПРОГОН НА ОДНОМ ПРИМЕРЕ (DEBUG)
    print("\n--- DEBUG: Проверка на одном примере ---")
    sample_text = test_data[0]['text'][:300]
    print(f"Текст: {sample_text}...")
    debug_preds = dp_model.extract(sample_text)
    print("Найденные сущности:", [(e.text, e.label) for e in debug_preds])
    print("----------------------------------------\n")

    # 4. Основной цикл
    evaluator = NEREvaluator()
    all_true = []
    dp_preds = []

    print(f">>> Обработка {len(test_data)} документов...")
    for doc in tqdm(test_data, desc="Inference"):
        all_true.extend(doc['entities'])
        dp_preds.extend(dp_model.extract(doc['text']))

    # 5. Результаты
    print("\n" + "="*40)
    print(" РЕЗУЛЬТАТЫ (RuBERT Collection3) ")
    print("="*40)

    res = evaluator.evaluate(all_true, dp_preds)

    # Strict
    s = res['strict']['overall']
    print(f"[Strict Match]  F1: {s.f1:.2f} | P: {s.precision:.2f} | R: {s.recall:.2f}")

    # Partial
    p = res['partial']['overall']
    print(f"[Partial Match] F1: {p.f1:.2f} (С учетом ошибок границ)")

    print("\nПо классам (Strict F1):")
    for tag, metrics in res['strict']['per_class'].items():
        print(f"  {tag:<4}: {metrics.f1:.2f} (Support: {metrics.support})")

# Запуск
if __name__ == "__main__":
    try:
        run_improved_comparison()
    except Exception as e:
        print(f"Ошибка: {e}")

>>> Загрузка данных...
Найдено документов: 93


Loading NEREL:   0%|          | 0/93 [00:00<?, ?it/s]


>>> Инициализация модели DeepPavlov (Collection3)...
Загрузка модели viktoroo/sberbank-rubert-base-collection3...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

...

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



--- DEBUG: Проверка на одном примере ---
Текст: Умер раненный в Афганистане американский военный
Американские военные в провинции Бадгис, Афганистан, 2011 год
Американский военный умер от ранений, полученных во время боя в Афганистане. Об этом 18 января сообщило Министерство обороны США.

26-летний сержант Кэмерон Мэддок (Cameron A. Meddock) из С...
Найденные сущности: [('афганистане', 'LOC'), ('бадгис', 'LOC'), ('афганистан', 'LOC'), ('афганистане', 'LOC'), ('министерство обороны', 'ORG'), ('сша', 'LOC'), ('кэмерон мэддок ( cameron a. meddock )', 'PER')]
----------------------------------------

>>> Обработка 20 документов...


Inference:   0%|          | 0/20 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



 РЕЗУЛЬТАТЫ (RuBERT Collection3) 
[Strict Match]  F1: 0.75 | P: 0.78 | R: 0.73
[Partial Match] F1: 0.82 (С учетом ошибок границ)

По классам (Strict F1):
  LOC : 0.75 (Support: 190)
  ORG : 0.58 (Support: 141)
  PER : 0.86 (Support: 207)


In [41]:
API_KEY = "sk-kDGfybeelE64OS3aSPydFw"

In [42]:
import json
import openai
import copy
from tqdm.auto import tqdm
import pandas as pd
import re
import os

# ==========================================
# 1. ПОДГОТОВКА ДАННЫХ (FIX NameError)
# ==========================================

# Загружаем наш dataset.json (созданный на этапе конвертации NEREL)
try:
    with open('dataset.json', 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print(f"Загружено документов: {len(raw_data)}")
except FileNotFoundError:
    print("Файл dataset.json не найден. Пожалуйста, выполните код конвертации из предыдущего шага.")
    raw_data = []

# Преобразуем в формат словаря {text: labels}, как в IE.ipynb
data_dict = {}
for item in raw_data:
    text = item['text']
    labels = item['labels']
    # В IE.ipynb ключи были: Кто, Тип вложения, Цель, Регион, Стоимость, Дата
    # У нас в dataset.json они уже такие (после маппинга)
    data_dict[text] = labels

# Разбиваем на примеры (few-shot) и валидацию (test)
# Берем первые 3 примера для обучения, остальные для теста
items = list(data_dict.items())
examples_dataset_raw = dict(items[:3])
validation = dict(items[3:23]) # Берем 20 штук для теста (чтобы не ждать долго)

# Формируем список примеров для промпта (формат для функции extract)
examples_dataset = []
for txt, lbl in examples_dataset_raw.items():
    examples_dataset.append({
        "context": txt,
        "labels": lbl,
        "question": "Извлеки сущности" # Заглушка, если требуется структурой
    })

print(f"Примеров для few-shot: {len(examples_dataset)}")
print(f"Примеров для валидации: {len(validation)}")


# ==========================================
# 2. НАСТРОЙКА LLM (RUDN Proxy)
# ==========================================



# Настройка клиента по инструкции из RUDN_LLM_Proxy_GettingStarted.ipynb
client = openai.OpenAI(
    api_key=API_KEY,
    base_url="https://llm.buffedby.ai/v1"
)

# Используем модель Llama 3.3 70B Instruct
MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"


# ==========================================
# 3. ФУНКЦИИ ГЕНЕРАЦИИ И ПАРСИНГА
# ==========================================

def clean_json_response(response_text):
    """Очищает ответ от markdown-обертки ```json ... ```"""
    # Ищем текст между первыми { и последними }
    match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if match:
        json_str = match.group(0)
    else:
        json_str = response_text

    try:
        return json.loads(json_str)
    except:
        return {}

def extract_with_llama(text, few_shot_examples=None):
    """Отправляет запрос к LLM."""

    system_prompt = (
        "Ты — эксперт по NER. Твоя задача — извлечь сущности из текста в формате JSON.\n"
        "Схема JSON:\n"
        "{\n"
        '  "Кто": ["Организация или Персона"],\n'
        '  "Тип вложения": [],\n'
        '  "Цель": [],\n'
        '  "Регион": ["Локация"],\n'
        '  "Стоимость": ["Деньги"],\n'
        '  "Дата": ["Временной период"]\n'
        "}\n"
        "Если сущность не найдена, оставь пустой список []. Выводи только JSON."
    )

    messages = [{"role": "system", "content": system_prompt}]

    # Добавляем примеры (Few-Shot)
    if few_shot_examples:
        for ex in few_shot_examples:
            messages.append({"role": "user", "content": ex['context']})
            messages.append({"role": "assistant", "content": json.dumps(ex['labels'], ensure_ascii=False)})

    messages.append({"role": "user", "content": text})

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.0,
            max_tokens=1024
        )
        return clean_json_response(response.choices[0].message.content)
    except Exception as e:
        print(f"LLM Error: {e}")
        return {}


# ==========================================
# 4. ЗАПУСК ЭКСПЕРИМЕНТА
# ==========================================

def run_llm_experiment(validation_data, examples_pool, shots_count):
    print(f"\n--- Запуск режима: {shots_count}-shot ---")

    current_examples = examples_pool[:shots_count] if shots_count > 0 else None
    predictions = []
    references = []

    for text, labels in tqdm(validation_data.items(), desc="Processing"):
        # Инференс
        pred = extract_with_llama(text, current_examples)

        predictions.append(pred)
        references.append(labels)

    return predictions, references


# Запуск по очереди: 0-shot, 1-shot, 5-shot
results_metrics = {}

# Убедитесь, что compute_evaluation_metrics определена (из предыдущего кода)
# Если нет, раскомментируйте блок ниже:
"""
def compute_metrics(conf_matrix, levels, partial_weight=0.5):
    # (Ваша функция метрик из IE.ipynb)
    # ... упрощенная версия для примера ...
    return pd.DataFrame([{"F1": 0.0}]) # Заглушка
"""

# Если примеров мало, 5-shot возьмет сколько есть (например, 3)
shots_list = [0, 1] if len(examples_dataset) < 5 else [0, 1, 5]

for shots in shots_list:
    preds, refs = run_llm_experiment(validation, examples_dataset, shots)

    # Считаем метрики (функция должна быть определена ранее!)
    if 'compute_evaluation_metrics' in globals():
        _, metrics_df = compute_evaluation_metrics(preds, refs)
        print(f"Результаты {shots}-shot:")
        display

Загружено документов: 746
Примеров для few-shot: 3
Примеров для валидации: 20

--- Запуск режима: 0-shot ---


Processing:   0%|          | 0/20 [00:00<?, ?it/s]


--- Запуск режима: 1-shot ---


Processing:   0%|          | 0/20 [00:00<?, ?it/s]